# FedNova: Federated Normalized Averaging

Federated fraud detection on SAML-D using **FedNova** (Federated Normalized Averaging).

FedNova corrects for *objective inconsistency* caused by varying local computation across clients: each client's local update is normalized by its **effective local steps** (tau_effective) before aggregation, then rescaled by the global effective tau.

**FL strategy:** FedNova (MLP)  
**Tree strategies:** LightGBM · XGBoost (local ensemble via prediction averaging)  
**Dataset:** SAML-D (Synthetic AML)  
**Partition:** Dirichlet alpha sweep {0.05, 0.1, 0.5}  
**Weights saved to:** `weights_fednova/`

In [1]:
!pip install lightgbm xgboost imbalanced-learn -q

## Imports

In [2]:
import os, gc, time, warnings, random, json
from collections import defaultdict

import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    average_precision_score, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix, fbeta_score
)

import xgboost as xgb
import lightgbm as lgb

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

print("All imports OK")

All imports OK


## Config

In [3]:
# ================================================================
# CONFIG
# ================================================================
T0  = time.time()
OUT = '/kaggle/working' if os.path.exists('/kaggle') else '.'
WEIGHTS_DIR = 'weights_fednova'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

def elapsed(): return f"{(time.time()-T0)/60:.1f}m"
def flush(): gc.collect()

N_BANKS        = 4
ALPHAS         = [0.05, 0.1, 0.5]

FL_ROUNDS      = 20
LOCAL_EPOCHS   = 5
LR             = 0.001
FEDNOVA_RHO    = 0.9   # local SGD momentum coefficient (0 = no momentum)

TEST_FRAC      = 0.15
VAL_FRAC       = 0.15
MIN_FRAUD      = 5
THRESH_DEFAULT = 0.5
TUNE_THRESHOLD = True
THRESH_GRID    = np.arange(0.05, 0.95, 0.05)
AP_KS          = (50, 100, 200)

MLP_CAP  = 100_000
TREE_CAP = 200_000

HIGH_RISK_COUNTRIES = {'mexico', 'turkey', 'morocco', 'uae', 'iran',
                       'nigeria', 'russia', 'venezuela', 'north korea'}

print(f"Config OK | WEIGHTS_DIR={WEIGHTS_DIR} | FEDNOVA_RHO={FEDNOVA_RHO}")

Config OK | WEIGHTS_DIR=weights_fednova | FEDNOVA_RHO=0.9


## Data Loading & Preprocessing (SAML-D)

In [4]:
def find_saml():
    candidates = [
        '/kaggle/input/datasets/berkanoztas/synthetic-transaction-monitoring-dataset-aml/SAML-D.csv',
    ]
    for p in candidates:
        if os.path.exists(p): return p
    for root, _, files in os.walk('.'):
        for f in files:
            if f.upper().startswith('SAML') and f.endswith('.csv'):
                return os.path.join(root, f)
    if os.path.exists('/kaggle'):
        for root, _, files in os.walk('/kaggle/input'):
            for f in files:
                if f.upper().startswith('SAML') and f.endswith('.csv'):
                    return os.path.join(root, f)
    return None


def load_saml():
    path = find_saml()
    if not path:
        raise FileNotFoundError("SAML-D.csv not found")
    print(f"Loading SAML-D: {path}")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]
    print(f"  {len(df):,} rows | fraud: {df['Is_laundering'].sum():,} "
          f"({df['Is_laundering'].mean()*100:.3f}%)")
    return df


def preprocess_saml(df):
    """
    Returns X, y, typ, src, t_col.
    Features (10): amt_log, amt_round, amt_thresh, hr_sin, hr_cos, pt_enc,
                   sl_risk, rl_risk, cross_border, curr_mm
    t_col: numeric Time column used for temporal split only (not in X).
    """
    y   = df['Is_laundering'].astype(int).values
    typ = df['Laundering_type'].astype(str).fillna('Unknown').values
    src = np.full(len(df), 'saml', dtype=object)

    t_raw = pd.to_numeric(df['Time'], errors='coerce').fillna(0).values
    if np.ptp(t_raw) < 1e-6:
        t_col = np.arange(len(df), dtype=np.int64)
    else:
        t_col = t_raw.astype(np.float64)

    amt = df['Amount'].fillna(0).abs()
    amt_log   = np.log1p(amt).values
    amt_round = (amt % 1000 == 0).astype(int).values
    amt_thresh= (amt < 10000).astype(int).values

    t = pd.to_numeric(df['Time'], errors='coerce').fillna(0)
    hr_sin = np.sin(2 * np.pi * t / 86400).values
    hr_cos = np.cos(2 * np.pi * t / 86400).values

    le_p   = LabelEncoder()
    pt_enc = le_p.fit_transform(df['Payment_type'].astype(str).fillna('Unknown'))

    sl_risk = df['Sender_bank_location'].astype(str).str.lower().apply(
        lambda x: int(any(h in x for h in HIGH_RISK_COUNTRIES))).values
    rl_risk = df['Receiver_bank_location'].astype(str).str.lower().apply(
        lambda x: int(any(h in x for h in HIGH_RISK_COUNTRIES))).values
    cross_border = (df['Sender_bank_location'].astype(str) !=
                    df['Receiver_bank_location'].astype(str)).astype(int).values
    curr_mm = (df['Payment_currency'].astype(str) !=
               df['Received_currency'].astype(str)).astype(int).values

    X = np.stack([
        amt_log, amt_round, amt_thresh,
        hr_sin, hr_cos, pt_enc.astype(float),
        sl_risk, rl_risk, cross_border, curr_mm,
    ], axis=1).astype(np.float32)

    print(f"  SAML-D features: {X.shape}")
    return X, y, typ, src, t_col

## Dirichlet Partition + Temporal Split

In [5]:
def _allocate_fraud(n_fraud, n_banks, alpha):
    props = np.random.dirichlet([alpha] * n_banks)
    fcnts = (props * n_fraud).astype(int)
    diff = n_fraud - fcnts.sum()
    if diff > 0:
        for _ in range(diff): fcnts[np.argmin(fcnts)] += 1
    elif diff < 0:
        for _ in range(-diff): fcnts[np.argmax(fcnts)] -= 1
    assert fcnts.sum() == n_fraud
    return fcnts


def partition_dataset(X, y, typ, t_col, n_banks, alpha):
    np.random.seed(42)
    fraud_idx = np.where(y == 1)[0]; legit_idx = np.where(y == 0)[0]
    np.random.shuffle(fraud_idx);    np.random.shuffle(legit_idx)
    fcnts = _allocate_fraud(len(fraud_idx), n_banks, alpha)
    lper  = len(legit_idx) // n_banks
    banks = []; fp = lp = 0
    for i in range(n_banks):
        ln  = lper if i < n_banks - 1 else len(legit_idx) - lp
        idx = np.concatenate([fraud_idx[fp:fp+fcnts[i]], legit_idx[lp:lp+ln]])
        banks.append(_split(i, X[idx], y[idx], typ[idx], t_col[idx],
                            f'SAML-Bank{i}'))
        fp += fcnts[i]; lp += ln
    print(f"Partition [SAML] {n_banks} banks | alpha={alpha}:")
    for b in banks:
        print(f"  {b['name']:20s}: {b['n_total']:>8,} txns | "
              f"{b['n_fraud']:>5,} fraud ({b['fraud_frac']*100:.3f}%) | "
              f"train={b['n_train_fraud']} val={b['n_val_fraud']} test={b['n_test_fraud']} fraud")
    return banks


def _split(bid, X, y, typ, t, name):
    order = np.argsort(t, kind='stable')
    X, y, typ = X[order], y[order], typ[order]
    n = len(X)
    n_train = max(int(n * (1 - TEST_FRAC - VAL_FRAC)), 1)
    n_val   = max(int(n * VAL_FRAC), 1)
    if n_train + n_val >= n: n_train = max(n - 2, 1); n_val = 1
    Xtr, ytr, ttr = X[:n_train],              y[:n_train],              typ[:n_train]
    Xvl, yvl, tvl = X[n_train:n_train+n_val], y[n_train:n_train+n_val], typ[n_train:n_train+n_val]
    Xte, yte, tte = X[n_train+n_val:],        y[n_train+n_val:],        typ[n_train+n_val:]
    sc  = StandardScaler()
    Xtr = sc.fit_transform(Xtr).astype(np.float32)
    Xvl = sc.transform(Xvl).astype(np.float32)
    Xte = sc.transform(Xte).astype(np.float32)
    return dict(
        id=bid, name=name, source='saml',
        X_train=Xtr, y_train=ytr, typ_train=ttr,
        X_val=Xvl,   y_val=yvl,   typ_val=tvl,
        X_test=Xte,  y_test=yte,  typ_test=tte,
        n_fraud=int(y.sum()), n_total=len(y), fraud_frac=float(y.mean()),
        n_train_fraud=int(ytr.sum()), n_val_fraud=int(yvl.sum()),
        n_test_fraud=int(yte.sum()),
    )


def safe_smote(X, y):
    if y.sum() < 5 or len(X) < 20: return X, y
    try:
        k = min(5, int(y.sum()) - 1)
        Xs, ys = SMOTE(random_state=42, k_neighbors=k).fit_resample(X, y)
        return Xs.astype(np.float32), ys.astype(np.int64)
    except Exception:
        return X, y


def get_mlp_data(X, y, cap=MLP_CAP):
    if len(X) > cap:
        fi = np.where(y == 1)[0]; li = np.where(y == 0)[0]
        nf = min(len(fi), cap // 10); nl = min(len(li), cap - nf)
        np.random.shuffle(fi); np.random.shuffle(li)
        idx = np.concatenate([fi[:nf], li[:nl]]); np.random.shuffle(idx)
        X, y = X[idx], y[idx]
    return safe_smote(X, y)

## MLP Helpers

In [6]:
def make_mlp():
    return MLPClassifier(
        hidden_layer_sizes=(128, 64, 32), activation='relu',
        solver='adam', learning_rate_init=LR,
        max_iter=LOCAL_EPOCHS, random_state=42,
        warm_start=False, tol=1e-4)


def get_weights(m):
    return [c.copy() for c in m.coefs_] + [i.copy() for i in m.intercepts_]


def set_weights(m, w):
    n = len(m.coefs_)
    m.coefs_      = [x.copy() for x in w[:n]]
    m.intercepts_ = [x.copy() for x in w[n:]]


def init_mlp(X, y):
    if 0 not in y: X, y = np.vstack([X, X[0:1]]), np.append(y, 0)
    if 1 not in y: X, y = np.vstack([X, X[0:1]]), np.append(y, 1)
    m = make_mlp(); m.fit(X, y); return m


def clone_mlp(template, w):
    m = make_mlp()
    tiny = np.zeros((2, template.coefs_[0].shape[0]), dtype=np.float32)
    try: m.fit(tiny, np.array([0, 1]))
    except: pass
    set_weights(m, w); return m


def mlp_proba(m, X):
    try:    return m.predict_proba(X)[:, 1]
    except: return np.full(len(X), 0.5)


def save_weights(weights, path):
    """Save weight list as .npz."""
    np.savez(path, *weights)


def load_weights(path):
    d = np.load(path + '.npz')
    return [d[k] for k in sorted(d.files)]


print("MLP helpers ready")

MLP helpers ready


## Client Training Functions

In [7]:
def train_mlp_client(config, X_train, y_train, X_val, y_val,
                     global_weights=None):
    """
    Train a local MLP for one client. If global_weights provided, initialise
    the model from them (federated warm-start).
    Returns (model, local_weights, val_loss).
    """
    X_tr, y_tr = get_mlp_data(X_train, y_train)
    if len(np.unique(y_tr)) < 2:
        return None, None, None

    # Build a tiny initialisation context if needed
    init_X = np.vstack([X_tr[:10], X_tr[:10]])
    init_y = np.array([0, 1] + [int(y_tr[0])] * (len(init_X) - 2))
    m = init_mlp(init_X, init_y)

    if global_weights is not None:
        set_weights(m, global_weights)

    m.fit(X_tr, y_tr)
    lw = get_weights(m)

    # Compute val loss proxy (log-loss on val)
    try:
        probs = mlp_proba(m, X_val)
        eps = 1e-7
        val_loss = -float(np.mean(
            y_val * np.log(probs + eps) + (1 - y_val) * np.log(1 - probs + eps)
        ))
    except Exception:
        val_loss = float('inf')

    return m, lw, val_loss


def train_lgb_client(config, X_train, y_train, X_val, y_val):
    n0 = (y_train == 0).sum(); n1 = max((y_train == 1).sum(), 1)
    X_tr, y_tr = safe_smote(X_train, y_train)
    m = lgb.LGBMClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        scale_pos_weight=float(n0 / n1),
        subsample=0.8, colsample_bytree=0.8,
        device='cpu', random_state=42, verbose=-1)
    m.fit(X_tr, y_tr)
    try:
        probs = m.predict_proba(X_val)[:, 1]
        eps = 1e-7
        val_loss = -float(np.mean(
            y_val * np.log(probs + eps) + (1 - y_val) * np.log(1 - probs + eps)
        ))
    except Exception:
        val_loss = float('inf')
    return m, None, val_loss


def train_xgb_client(config, X_train, y_train, X_val, y_val):
    n0 = (y_train == 0).sum(); n1 = max((y_train == 1).sum(), 1)
    X_tr, y_tr = safe_smote(X_train, y_train)
    m = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        scale_pos_weight=float(n0 / n1),
        subsample=0.8, colsample_bytree=0.8,
        tree_method='hist', eval_metric='aucpr',
        use_label_encoder=False,
        random_state=42, verbosity=0)
    m.fit(X_tr, y_tr)
    try:
        probs = m.predict_proba(X_val)[:, 1]
        eps = 1e-7
        val_loss = -float(np.mean(
            y_val * np.log(probs + eps) + (1 - y_val) * np.log(1 - probs + eps)
        ))
    except Exception:
        val_loss = float('inf')
    return m, None, val_loss


print("Client training functions ready")

Client training functions ready


## FedNova Aggregation

In [8]:
def fednova_aggregate(global_weights, delta_list, sizes, tau_list):
    """
    FedNova aggregation: normalize each client's update by tau_effective,
    then compute tau_global-weighted sum.
    """
    if not delta_list:
        return global_weights
    total = sum(sizes)
    # Global effective tau (weighted average)
    tau_global = sum(t * s / total for t, s in zip(tau_list, sizes))
    n_layers = len(delta_list[0])
    update = [
        tau_global * sum(
            (s / total) * (d[i] / max(t, 1))
            for d, s, t in zip(delta_list, sizes, tau_list)
        )
        for i in range(n_layers)
    ]
    return [gw + u for gw, u in zip(global_weights, update)]


def fedavg_aggregate_probs(prob_list, sizes):
    total = sum(sizes)
    return sum((s / total) * p for p, s in zip(prob_list, sizes))


print("FedNova aggregation ready")

FedNova aggregation ready


## FL Round Functions

In [9]:
def run_fednova_round_mlp(client_data, val_splits, global_weights, global_model, config):
    delta_list, sizes, tau_list = [], [], []
    for idx in range(len(client_data)):
        X_c, y_c = client_data[idx]
        X_vl, y_vl = val_splits[idx]
        if len(np.unique(y_c)) < 2:
            continue
        m, w_dict, _ = train_mlp_client(config, X_c, y_c, X_vl, y_vl,
                                        global_weights=global_weights)
        if m is None:
            continue
        local_w = get_weights(m)
        delta = [lw - gw for lw, gw in zip(local_w, global_weights)]
        # tau = number of effective local update steps
        tau = LOCAL_EPOCHS * max(len(X_c) // 32, 1)
        delta_list.append(delta)
        sizes.append(len(X_c))
        tau_list.append(tau)
    return fednova_aggregate(global_weights, delta_list, sizes, tau_list)


def run_fednova_round_tree(client_data, val_splits, model_type, config):
    # Trees: prediction ensemble (normalization doesn't apply to trees)
    models = []
    for idx in range(len(client_data)):
        X_c, y_c = client_data[idx]
        X_vl, y_vl = val_splits[idx]
        if len(np.unique(y_c)) < 2:
            continue
        if model_type == 'lightgbm':
            m, _, _ = train_lgb_client(config, X_c, y_c, X_vl, y_vl)
        else:
            m, _, _ = train_xgb_client(config, X_c, y_c, X_vl, y_vl)
        models.append((m, len(X_c)))
    return models


print("FL round functions ready")

FL round functions ready


## Threshold Tuning & Metrics

In [10]:
def tune_threshold(y_val, probs_val, default=THRESH_DEFAULT):
    if not TUNE_THRESHOLD or y_val.sum() == 0 or y_val.sum() == len(y_val):
        return default
    best_t, best_f2 = default, -1
    for t in THRESH_GRID:
        preds = (probs_val >= t).astype(int)
        if preds.sum() == 0:
            continue
        f2 = fbeta_score(y_val, preds, beta=2, zero_division=0)
        if f2 > best_f2:
            best_f2, best_t = f2, float(t)
    return best_t


TYP_W = {
    'Cycle': 2.5, 'Bipartite': 2.5, 'Stacked Bipartite': 2.5,
    'Structuring': 2.5, 'Smurfing': 2.0, 'Scatter-Gather': 2.0,
    'Gather-Scatter': 2.0, 'Fan_Out': 2.0, 'Fan_In': 2.0,
    'Layered_Fan_Out': 2.0, 'Layered_Fan_In': 2.0,
    'Deposit-Send': 2.0, 'Over-Invoicing': 2.0,
}


def compute_metrics(y_true, probs, typ=None, thresh=THRESH_DEFAULT):
    preds = (probs >= thresh).astype(int)
    n_pos = int(y_true.sum())
    tn = int(((y_true == 0) & (preds == 0)).sum())
    fp = int(((y_true == 0) & (preds == 1)).sum())
    specificity = tn / max(tn + fp, 1)
    fpr = fp / max(tn + fp, 1)

    if n_pos == 0:
        apk = {f'ap_at_{k}': 0. for k in AP_KS}
        return {'f1': 0., 'precision': 0., 'recall': 0., 'auprc': 0.,
                'mcc': 0., 'f2': 0., **apk,
                'typ_coverage': 0., 'typ_wf1': 0.,
                'specificity': float(specificity), 'fpr': float(fpr),
                'false_positives': fp, 'n_test_fraud': 0, 'threshold': thresh}

    f1   = f1_score(y_true, preds, zero_division=0)
    prec = precision_score(y_true, preds, zero_division=0)
    rec  = recall_score(y_true, preds, zero_division=0)
    try:    auprc = average_precision_score(y_true, probs)
    except: auprc = float(y_true.mean())
    mcc = matthews_corrcoef(y_true, preds) if preds.sum() > 0 else 0.
    b2 = 4; d = b2 * prec + rec
    f2 = (1 + b2) * prec * rec / d if d > 0 else 0.
    sidx = np.argsort(probs)[::-1]
    apk  = {f'ap_at_{k}': float(y_true[sidx[:min(k, len(y_true))]].sum() /
                                 max(min(k, len(y_true)), 1)) for k in AP_KS}
    typ_cov = 0.; typ_wf1 = 0.
    if typ is not None:
        laund = [t for t in np.unique(typ) if t not in ('Unknown', 'Normal_Cash_Deposits')]
        if laund:
            typ_cov = sum(
                1 for t in laund
                if ((typ == t) & (y_true == 1) & (preds == 1)).sum() > 0
            ) / len(laund)
        ws = wt = 0.
        for t in np.unique(typ):
            if t in ('Unknown', 'Normal_Cash_Deposits'): continue
            mask = typ == t
            if mask.sum() < 2 or y_true[mask].sum() == 0: continue
            f = f1_score(y_true[mask], preds[mask], zero_division=0)
            w = TYP_W.get(t, 1.5); ws += w * f; wt += w
        typ_wf1 = ws / max(wt, 1e-8)

    return {'f1': f1, 'precision': prec, 'recall': rec, 'auprc': auprc,
            'mcc': mcc, 'f2': f2, **apk,
            'typ_coverage': typ_cov, 'typ_wf1': typ_wf1,
            'specificity': float(specificity), 'fpr': float(fpr),
            'false_positives': fp, 'n_test_fraud': n_pos, 'threshold': thresh}


def agg(ml):
    if not ml: return {}
    return {k: round(float(np.mean([m[k] for m in ml])), 4) for k in ml[0]}


def fairness(f1s, lf=None):
    r = dict(client_equity=round(float(np.std(f1s)), 4),
             worst_bank_f1=round(float(min(f1s)), 4),
             best_bank_f1=round(float(max(f1s)), 4))
    if lf and len(lf) == len(f1s):
        r['collab_gain'] = round(float(np.mean([a - b for a, b in zip(f1s, lf)])), 4)
    return r


print("Metrics ready")

Metrics ready


## FedNova Training Sweep

In [11]:
def run_fednova_fl(banks, config):
    """
    Full FedNova FL training over FL_ROUNDS rounds.
    Returns (global_model, fl_history).
    """
    print(f"  [FedNova] {FL_ROUNDS} rounds", end='  ', flush=True)

    # Initialise global model
    init_X = np.vstack([b['X_train'][:50] for b in banks])
    init_y = np.concatenate([b['y_train'][:50] for b in banks])
    global_model = init_mlp(init_X, init_y)
    global_weights = get_weights(global_model)

    # Build per-client (data, val) lists
    client_data = [(b['X_train'], b['y_train']) for b in banks]
    val_splits  = [(b['X_val'],   b['y_val'])   for b in banks]

    hist = []
    for rnd in range(1, FL_ROUNDS + 1):
        global_weights = run_fednova_round_mlp(
            client_data, val_splits, global_weights, global_model, config
        )

        if rnd % 5 == 0 or rnd == FL_ROUNDS:
            print(f"{rnd}", end=' ', flush=True)
            gm = clone_mlp(global_model, global_weights)
            for b in banks:
                yt = b['y_test']
                probs = mlp_proba(gm, b['X_test'])
                preds = (probs >= THRESH_DEFAULT).astype(int)
                f1_val    = f1_score(yt, preds, zero_division=0) if yt.sum() > 0 else 0.
                auprc_val = average_precision_score(yt, probs)   if yt.sum() > 0 else 0.
                hist.append(dict(
                    round=rnd, bank_id=b['id'], bank_name=b['name'],
                    strategy='fednova', source='saml',
                    f1=f1_val, auprc=auprc_val,
                    n_test_fraud=int(yt.sum())
                ))
        flush()

    print()
    final_model = clone_mlp(global_model, global_weights)
    return final_model, global_weights, hist


def run_tree_ensemble(banks, model_type, config):
    """
    Train one tree model per client, return list of (model, n_samples).
    Trees: prediction ensemble (normalization doesn't apply to trees).
    """
    print(f"  [FedNova-{model_type}] training per-client trees...", end=' ')
    client_data = [(b['X_train'], b['y_train']) for b in banks]
    val_splits  = [(b['X_val'],   b['y_val'])   for b in banks]
    models = run_fednova_round_tree(client_data, val_splits, model_type, config)
    print(f"{len(models)} models trained")
    return models


def predict_tree_ensemble(models, X):
    """Weighted average prediction from tree ensemble."""
    if not models:
        return np.full(len(X), 0.5)
    prob_list = [m.predict_proba(X)[:, 1] for m, _ in models]
    sizes     = [s for _, s in models]
    return fedavg_aggregate_probs(prob_list, sizes)


print("FedNova training functions ready")

FedNova training functions ready


## Evaluation

In [12]:
def evaluate_fednova(banks, mlp_model, lgb_models, xgb_models):
    """
    Evaluate FedNova MLP + LightGBM ensemble + XGBoost ensemble on each bank's test set.
    """
    rows = []
    total_fraud = sum(int(b['y_test'].sum()) for b in banks)

    strategies = [
        ('fednova_mlp',     'fl'),
        ('fednova_lgb',     'tree_ensemble'),
        ('fednova_xgb',     'tree_ensemble'),
    ]

    for strat, mtype in strategies:
        bm = []
        for b in banks:
            yt  = b['y_test']
            tte = b.get('typ_test')

            if strat == 'fednova_mlp':
                pv = mlp_proba(mlp_model, b['X_val'])
                pt = mlp_proba(mlp_model, b['X_test'])
            elif strat == 'fednova_lgb':
                pv = predict_tree_ensemble(lgb_models, b['X_val'])
                pt = predict_tree_ensemble(lgb_models, b['X_test'])
            else:  # xgb
                pv = predict_tree_ensemble(xgb_models, b['X_val'])
                pt = predict_tree_ensemble(xgb_models, b['X_test'])

            t_tune = tune_threshold(b['y_val'], pv)
            bm.append(compute_metrics(yt, pt, tte, thresh=t_tune))

        a  = agg(bm)
        fa = fairness([m['f1'] for m in bm])
        n_with_fraud = sum(1 for b in banks if b['y_test'].sum() > 0)
        rows.append({
            'strategy':           strat,
            'model_type':         mtype,
            **a, **fa,
            'n_eval_banks':       len(banks),
            'n_banks_with_fraud': n_with_fraud,
            'total_test_fraud':   total_fraud,
        })
    return rows


print("Evaluation function ready")

Evaluation function ready


## Plotting

In [13]:
COLORS = {
    'fednova_mlp': '#854F0B',
    'fednova_lgb': '#D85A30',
    'fednova_xgb': '#A32D2D',
}


def plot_fednova_results(df_bm, fl_hist, tag):
    strats = sorted(df_bm['strategy'].unique())
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(
        f'FedNova — {tag} — SAML-D · Temporal Split · Dirichlet\n'
        'FedNova MLP (normalized aggregation) + LightGBM ensemble + XGBoost ensemble',
        fontsize=10, fontweight='bold'
    )

    # Left: primary metrics bar chart
    ax = axes[0]; x = np.arange(len(strats)); w = 0.22
    for mi, (m, lbl, c) in enumerate([
        ('auprc', 'AUPRC', '#2E4057'),
        ('f2',    'F2',    '#048A81'),
        ('mcc',   'MCC',   '#54C6EB')
    ]):
        vals = [float(df_bm[df_bm['strategy'] == s][m].mean()) for s in strats]
        ax.bar(x + (mi - 1) * w, vals, w, label=lbl, color=c, alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(strats, fontsize=8, rotation=30, ha='right')
    ax.set_ylim(0, 1); ax.legend(fontsize=8, frameon=False)
    ax.set_title('Primary Metrics (avg all banks)', fontsize=9, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='y', alpha=0.25)

    # Centre: AUPRC vs Recall
    ax = axes[1]
    for si, s in enumerate(strats):
        c  = COLORS.get(s, '#999')
        ap = float(df_bm[df_bm['strategy'] == s]['auprc'].mean())
        rc = float(df_bm[df_bm['strategy'] == s]['recall'].mean())
        ax.bar(si - 0.18, ap, 0.35, color=c, alpha=0.85)
        ax.bar(si + 0.18, rc, 0.35, color=c, alpha=0.4,
               edgecolor=c, linewidth=1.5)
    ax.set_xticks(range(len(strats)))
    ax.set_xticklabels(strats, fontsize=8, rotation=30, ha='right')
    ax.set_ylim(0, 1)
    ax.set_title('AUPRC (solid) vs Recall (hollow)', fontsize=9, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='y', alpha=0.25)

    # Right: FedNova MLP learning curve
    ax = axes[2]
    dfh = pd.DataFrame(fl_hist)
    if not dfh.empty:
        g = dfh.groupby('round')['auprc'].mean().reset_index()
        ax.plot(g['round'], g['auprc'], label='FedNova MLP',
                color=COLORS['fednova_mlp'], lw=2, marker='o', ms=4)
    ax.set_xlabel('FL Round'); ax.set_ylim(0, 1)
    ax.set_title('FedNova Learning Curve (avg AUPRC)', fontsize=9, fontweight='bold')
    ax.legend(fontsize=8, frameon=False)
    ax.spines[['top', 'right']].set_visible(False); ax.grid(alpha=0.25)

    plt.tight_layout()
    p = f'{WEIGHTS_DIR}/{tag}_fednova_results.png'
    plt.savefig(p, dpi=150, bbox_inches='tight'); plt.close()
    print(f"Saved: {p}")


def print_fednova_summary(df_bm, label):
    print(f"\n{'='*70}")
    print(f"FedNova — {label} — SAML-D · Temporal Split · Dirichlet")
    print('=' * 70)
    cols = ['strategy', 'auprc', 'mcc', 'f2', 'f1', 'recall',
            'specificity', 'fpr', 'ap_at_100', 'typ_wf1',
            'client_equity', 'worst_bank_f1',
            'threshold', 'n_eval_banks', 'n_banks_with_fraud', 'total_test_fraud']
    avail = [c for c in cols if c in df_bm.columns]
    print(df_bm[avail].sort_values('auprc', ascending=False).to_string(index=False))
    best = df_bm.loc[df_bm['auprc'].idxmax()]
    print(f"\nBEST: {best['strategy']}  AUPRC={best['auprc']:.4f}  "
          f"MCC={best['mcc']:.4f}  F2={best['f2']:.4f}")


print("Plotting functions ready")

Plotting functions ready


## Main: FedNova Alpha Sweep

In [14]:
# ================================================================
# MAIN: FedNova sweep — SAML-D × 3 Dirichlet alphas
# ================================================================
print("Loading SAML-D dataset...")
df_raw = load_saml()
X, y, typ, src, t_col = preprocess_saml(df_raw)
del df_raw; flush()
print(f"Preprocessing done | {elapsed()}")

config = {}   # passed through to training helpers
all_combined = []

for alpha in ALPHAS:
    tag = f"saml_alpha{alpha}"
    print(f"\n{'='*55}")
    print(f"  FedNova | SAML-D | alpha={alpha} | {N_BANKS} banks")
    print(f"{'='*55}")

    banks = partition_dataset(X, y, typ, t_col, N_BANKS, alpha)

    # ── FedNova MLP ──
    print("\nFedNova MLP training:")
    mlp_model, global_weights, fl_hist = run_fednova_fl(banks, config)

    # Save final MLP weights
    w_path = os.path.join(WEIGHTS_DIR, f'{tag}_fednova_mlp')
    save_weights(global_weights, w_path)
    print(f"  Weights saved: {w_path}.npz")

    for h in fl_hist:
        h['alpha'] = alpha

    # ── LightGBM ensemble ──
    print("\nFedNova LightGBM ensemble:")
    lgb_models = run_tree_ensemble(banks, 'lightgbm', config)

    # ── XGBoost ensemble ──
    print("FedNova XGBoost ensemble:")
    xgb_models = run_tree_ensemble(banks, 'xgboost', config)

    # ── Evaluate ──
    print("\nEvaluating...")
    bm_rows = evaluate_fednova(banks, mlp_model, lgb_models, xgb_models)
    for r in bm_rows:
        r['alpha'] = alpha
        r['dataset'] = 'SAML'

    df_bm = pd.DataFrame(bm_rows)
    df_bm.to_csv(f'{WEIGHTS_DIR}/{tag}_fednova_benchmark.csv', index=False)
    pd.DataFrame(fl_hist).to_csv(f'{WEIGHTS_DIR}/{tag}_fednova_fl_history.csv', index=False)

    print_fednova_summary(df_bm, f"alpha={alpha}")
    plot_fednova_results(df_bm, fl_hist, tag)

    all_combined.append(df_bm)
    del banks, mlp_model, lgb_models, xgb_models, fl_hist; flush()
    print(f"  Done | {elapsed()}")

# ── Consolidated output ──
combined = pd.concat(all_combined, ignore_index=True)
combined.to_csv(f'{WEIGHTS_DIR}/fednova_saml_combined.csv', index=False)

print(f"\n{'='*55}")
print("FEDNOVA SWEEP COMPLETE")
print(f"{'='*55}")
print(f"Total runtime: {elapsed()}")
print(f"\nOutputs in '{WEIGHTS_DIR}':")
for f in sorted(os.listdir(WEIGHTS_DIR)):
    print(f"  {f}")

Loading SAML-D dataset...
Loading SAML-D: /kaggle/input/datasets/berkanoztas/synthetic-transaction-monitoring-dataset-aml/SAML-D.csv
  9,504,852 rows | fraud: 9,873 (0.104%)
  SAML-D features: (9504852, 10)
Preprocessing done | 1.3m

  FedNova | SAML-D | alpha=0.05 | 4 banks
Partition [SAML] 4 banks | alpha=0.05:
  SAML-Bank0          : 2,373,854 txns |   110 fraud (0.005%) | train=78 val=12 test=20 fraud
  SAML-Bank1          : 2,373,745 txns |     1 fraud (0.000%) | train=1 val=0 test=0 fraud
  SAML-Bank2          : 2,383,506 txns | 9,762 fraud (0.410%) | train=6688 val=1397 test=1677 fraud
  SAML-Bank3          : 2,373,747 txns |     0 fraud (0.000%) | train=0 val=0 test=0 fraud

FedNova MLP training:
  [FedNova] 20 rounds  5 10 15 20 
  Weights saved: weights_fednova/saml_alpha0.05_fednova_mlp.npz

FedNova LightGBM ensemble:
  [FedNova-lightgbm] training per-client trees... 3 models trained
FedNova XGBoost ensemble:
  [FedNova-xgboost] training per-client trees... 3 models trained


## Consolidated Results Summary

In [15]:
combined = pd.read_csv(f'{WEIGHTS_DIR}/fednova_saml_combined.csv')

print("FedNova — SAML-D · All Alphas · AUPRC Pivot")
piv = combined.pivot_table(
    index='alpha',
    columns='strategy',
    values='auprc',
    aggfunc='mean'
).round(4)
print(piv.to_string())

print("\nFedNova — SAML-D · All Alphas · MCC Pivot")
piv_mcc = combined.pivot_table(
    index='alpha',
    columns='strategy',
    values='mcc',
    aggfunc='mean'
).round(4)
print(piv_mcc.to_string())

print("\nFedNova — SAML-D · All Alphas · F2 Pivot")
piv_f2 = combined.pivot_table(
    index='alpha',
    columns='strategy',
    values='f2',
    aggfunc='mean'
).round(4)
print(piv_f2.to_string())

FedNova — SAML-D · All Alphas · AUPRC Pivot
strategy  fednova_lgb  fednova_mlp  fednova_xgb
alpha                                          
0.05           0.0078       0.0230       0.0059
0.10           0.0454       0.0328       0.0477
0.50           0.0676       0.0261       0.0358

FedNova — SAML-D · All Alphas · MCC Pivot
strategy  fednova_lgb  fednova_mlp  fednova_xgb
alpha                                          
0.05           0.0043       0.0216       0.0060
0.10           0.0085       0.0350       0.0094
0.50           0.0113       0.0348       0.0114

FedNova — SAML-D · All Alphas · F2 Pivot
strategy  fednova_lgb  fednova_mlp  fednova_xgb
alpha                                          
0.05           0.0071       0.0259       0.0068
0.10           0.0068       0.0447       0.0071
0.50           0.0078       0.0362       0.0077
